In [3]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

C:\Users\wang\AppData\Local\Temp\ipykernel_20740\3336491994.py:7: DeprecationWarning: 'asyncio.get_event_loop_policy' is deprecated and slated for removal in Python 3.16
  if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
C:\Users\wang\AppData\Local\Temp\ipykernel_20740\3336491994.py:7: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):


In [8]:
# 1.加载环境变量
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv()

# 2.初始化模型
model = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

# 3.定义工具，用MCP获取工具

# 3.1.定义mcp client
client = MultiServerMCPClient(
    {
        "time-mcp": {
            "transport": "stdio",
            "args": [
                "-y",
                "time-mcp"
            ],
            "command": "npx"
        },
        "12306-mcp": {
          "transport": "streamable_http",
          "url": "https://mcp.api-inference.modelscope.net/bae4d459a7284f/mcp"
        }
    }
)
# 3.2.用client拉取tool
tools = await client.get_tools()


# 4.创建Agent，绑定模型和工具
agent = create_agent(
    model=model,
    tools=tools
)

# 5.由于MCP的Tool是异步的，所以必须用ainvoke调用Agent，是异步调用
response = await agent.ainvoke(
    {"messages": [HumanMessage("查询一下合肥到上海的高铁信息")]}
)
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

查询一下合肥到上海的高铁信息
================================== Ai Message ==================================

我来帮您查询合肥到上海的高铁信息。首先获取当前日期和车站信息。
Tool Calls:
  get-current-date (call_00_PHIsapXl1RkVhJdSHn696263)
 Call ID: call_00_PHIsapXl1RkVhJdSHn696263
  Args:
  get-station-code-of-citys (call_01_mMY1DL6oxmthQgAoyFdA7336)
 Call ID: call_01_mMY1DL6oxmthQgAoyFdA7336
  Args:
    citys: 合肥|上海
================================= Tool Message =================================
Name: get-current-date

[{'type': 'text', 'text': '2026-11-18', 'id': 'lc_93bfc2fe-22b2-48e0-9ad5-d89b1236c553'}]
================================= Tool Message =================================
Name: get-station-code-of-citys

[{'type': 'text', 'text': '{"合肥":{"station_code":"HFH","station_name":"合肥"},"上海":{"station_code":"SHH","station_name":"上海"}}', 'id': 'lc_96a00e04-d233-48f1-808c-dd7c595d73dc'}]
================================== Ai Message =========